#**Importación, inspección y limpieza básica**

Para realizar análisis estadísticos, siempre es necesario trabajar con datos. Estos datos generalmente son recolectados a través de encuestas, experimentos o sistemas de registro, y muchas veces se almacenan en bases de datos grandes o archivos estructurados. Capturar esta información manualmente en R resultaría poco práctico, propenso a errores y muy tardado. Por ello, es fundamental contar con herramientas que permitan importar los datos de forma eficiente directamente al entorno de trabajo.
La importación de datos es uno de los primeros pasos fundamentales en cualquier análisis estadístico en R. Consiste en cargar información desde distintas fuentes —como archivos de texto, hojas de cálculo de Excel, bases de datos o incluso páginas web— hacia el entorno de trabajo de R para su posterior procesamiento y análisis.
R ofrece múltiples funciones para esta tarea. Por ejemplo, read.csv() y read.table() permiten importar archivos delimitados, mientras que paquetes como readr y data.table proporcionan alternativas más rápidas y eficientes. Para archivos de Excel, el paquete readxl facilita la lectura de hojas sin necesidad de software adicional. Además, R puede conectarse a bases de datos mediante paquetes como DBI, lo que amplía su capacidad para trabajar con grandes volúmenes de información.
Una correcta importación implica no solo cargar los datos, sino también verificar su estructura, tipos de variables y posibles inconsistencias. Esto garantiza que el análisis posterior sea confiable y reproducible.
En resumen, dominar la importación de datos en R permite al usuario integrar información de diversas fuentes y constituye la base para cualquier flujo de trabajo en ciencia de datos.

In [ ]:
#Cargamos librerias (las primeras tres solo para los que trabajan en colab)
# install.packages("tidyverse")
# install.packages("janitor")
# install.packages("skimr")
library(tidyverse)
library(janitor)
library(skimr)

In [ ]:
datos <- read_csv("data/datos_medicos.csv")

dir.create("resultados", showWarnings = FALSE)

# Ver primeras filas
head(datos)

# Estructura
str(datos)

# Resumen general
summary(datos)

# Resumen compacto con skimr
skim(datos)


#**Limpiar nombres de variables**

La limpieza de los nombres de las variables es un paso esencial en el análisis de datos en R. Cuando los datos se importan desde archivos externos, es común que los nombres de las columnas contengan espacios, caracteres especiales, abreviaturas poco claras o formatos inconsistentes. Por ejemplo, variables como glucosa_mg_dl o hb_pre pueden no ser inmediatamente comprensibles o pueden dificultar la escritura de código.
Tener nombres de variables claros, consistentes y estandarizados facilita la interpretación de los datos y mejora la legibilidad del código, especialmente cuando se trabaja en equipo o se desarrollan análisis reproducibles. Además, evitar espacios o caracteres especiales previene errores al programar y simplifica el uso de funciones y paquetes en R.
Por ello, es recomendable transformar los nombres a formatos simples, como usar minúsculas, separar palabras con guiones bajos y emplear términos descriptivos (por ejemplo, glucosa en lugar de abreviaturas innecesarias si el contexto ya está definido).

In [ ]:
datos <- datos %>%
  clean_names()

names(datos)


#**Categorizar**

Importancia de convertir los datos a variables categóricas en R
En el análisis de datos es fundamental identificar correctamente el tipo de cada variable. Muchas veces, al importar datos en R, variables que representan categorías —como sexo, grupo de tratamiento, estado de fumador o presencia de enfermedad— son leídas como texto o números, cuando en realidad deben tratarse como variables categóricas (factores).
Convertir estas variables a formato categórico es importante porque permite a R interpretarlas correctamente en los análisis estadísticos. Por ejemplo, variables como sexo, grupo o fumador no representan cantidades numéricas, sino categorías, y tratarlas como números puede producir resultados incorrectos o interpretaciones erróneas.
Además, muchos modelos estadísticos y funciones en R (como regresión, tablas de frecuencia o gráficos) dependen de que las variables categóricas estén definidas adecuadamente. Esto asegura que los niveles de cada categoría se manejen correctamente y que los resultados sean más claros e interpretables.

In [ ]:
datos <- datos %>%
  mutate(
    sexo = factor(sexo),
    grupo = factor(grupo),
    fumador = factor(fumador),
    diabetes = factor(diabetes, levels = c(0, 1), labels = c("No", "Sí")),
    hipertension = factor(hipertension, levels = c(0, 1), labels = c("No", "Sí")),
    evento_complicacion = factor(evento_complicacion, levels = c(0, 1), labels = c("No", "Sí"))
  )

str(datos)

#**Valores perdidos en R**

En el análisis de datos es muy común encontrar valores perdidos, también conocidos como missing values o NA en R. Estos aparecen cuando no se registró información, hubo errores en la captura o simplemente el dato no estaba disponible. Ignorar estos valores puede afectar los resultados del análisis, por lo que es importante identificarlos y cuantificarlos.
En R, podemos detectar fácilmente cuántos valores faltantes hay en cada variable usando la función:

> colSums(is.na(datos))



Aquí, is.na(datos) convierte todos los valores del dataset en TRUE (si falta el dato) o FALSE (si está presente), y colSums() suma los TRUE por columna, dándonos el número total de valores perdidos en cada variable.
Sin embargo, además del número absoluto, es muy útil conocer el porcentaje de datos faltantes, ya que esto nos permite evaluar su impacto relativo. Para ello usamos el siguiente código:


In [ ]:
colSums(is.na(datos))

# Porcentaje de faltantes por variable
faltantes <- datos %>%
  summarise(across(everything(), ~ mean(is.na(.)) * 100)) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "porcentaje_faltante") %>%
  arrange(desc(porcentaje_faltante))

faltantes


Este bloque hace lo siguiente:

mean(is.na(.)) * 100: calcula el porcentaje de valores faltantes por variable.
summarise(across(...)): aplica ese cálculo a todas las columnas.
pivot_longer(): transforma los resultados a un formato más ordenado (una fila por variable).
arrange(desc(...)): ordena las variables de mayor a menor porcentaje de faltantes.

De esta manera, podemos identificar rápidamente qué variables tienen más datos incompletos y decidir qué hacer: eliminarlas, imputar valores o analizarlas con precaución.
En resumen, conocer y cuantificar los valores perdidos es un paso clave para asegurar la calidad y validez del análisis estadístico.

#**Crear variables derivadas**

En el análisis de datos, además de trabajar con las variables originales, también es muy común crear nuevas variables a partir de la información existente. A estas se les conoce como variables derivadas, y permiten resumir, transformar o reinterpretar los datos para facilitar su análisis clínico o estadístico.
Por ejemplo, en lugar de analizar directamente el índice de masa corporal (IMC) como un número continuo, podemos crear una variable categórica que indique si un paciente presenta obesidad o no. De igual forma, podemos transformar valores de glucosa en una variable que indique si está elevada, o calcular cambios entre dos mediciones, como la diferencia entre hemoglobina antes y después de un tratamiento.
Esto se realiza en R usando la función mutate(), que nos permite crear nuevas columnas en el conjunto de datos:

In [11]:
datos <- datos %>%
  mutate(
    obesidad = case_when(
      is.na(imc) ~ NA_character_,
      imc >= 30 ~ "Sí",
      imc < 30 ~ "No"
    ),
    glucosa_alta = case_when(
      is.na(glucosa_mg_dl) ~ NA_character_,
      glucosa_mg_dl >= 126 ~ "Sí",
      glucosa_mg_dl < 126 ~ "No"
    ),
    cambio_hb = hb_post - hb_pre
  ) %>%
  mutate(
    obesidad = factor(obesidad, levels = c("No", "Sí")),
    glucosa_alta = factor(glucosa_alta, levels = c("No", "Sí"))
  )

En este bloque:

case_when() permite definir condiciones para clasificar los datos.
Se respeta la presencia de valores perdidos (NA) para no introducir sesgos.
Se crean variables categóricas como obesidad y glucosa_alta.
También se genera una variable numérica (cambio_hb) que representa el cambio entre dos mediciones.

Después, convertimos las variables categóricas a factores:


Esto es importante para que R las interprete correctamente en análisis estadísticos y gráficos.
Finalmente, podemos revisar las nuevas variables:

In [ ]:
datos %>%
  select(id, imc, obesidad, glucosa_mg_dl, glucosa_alta, hb_pre, hb_post, cambio_hb) %>%
  head(10)

#**Guardar la base de datos limpia en R**

Una vez que hemos importado, limpiado, transformado y preparado nuestros datos, es fundamental guardar la base final. Esto permite conservar todo el trabajo realizado y facilita su uso posterior, ya sea para análisis adicionales, compartir con otros investigadores o documentar el proceso.
En R, podemos guardar nuestro conjunto de datos en formato .csv utilizando la función:


In [15]:
write_csv(datos, "resultados/datos_medicos_limpios.csv")

Esta línea hace lo siguiente:

write_csv(): exporta el dataset a un archivo CSV.
datos: es la base ya limpia y transformada.
"resultados/datos_medicos_limpios.csv": indica la ruta y el nombre del archivo que se va a generar.

Es importante notar que:

El archivo se guardará en la carpeta especificada (en este caso, resultados).
Si la carpeta no existe, R puede generar un error, por lo que debe crearse previamente.
Guardar versiones limpias de los datos ayuda a mantener un flujo de trabajo ordenado y reproducible.